<a href="https://colab.research.google.com/github/KudjoJedidiah/Google_deepmind/blob/main/Ngrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GOAL OF THIS LAB:
Automatically estimate the probabilities for predicting the next word and build an n-gram model using a small dataset of paragraphs

What I am supposed to learn by the end of this lab
1. Split paragraphs in a dataset into word-like units called tokens, a process known as tokenization.
2. Estimate the probabilities for an n-gram language model from a dataset.
3. Use the n-gram language model to predict individual tokens and longer continuations.

In [ ]:
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"


  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-oy3v5su1
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-oy3v5su1
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 5338de78c28ecd87548a3b003038b095abf5cb93
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.9/533.9 kB 23.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 113.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.3/609.3 kB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 MB 48.7 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import random
from collections import Counter,defaultdict

import textwrap
import pandas as pd
from ai_foundations.feedback.course_1 import ngrams

In [ ]:
africa_galore = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
)
dataset=africa_galore['description']
print(f"The dataset consists of {dataset.shape[0]} paragraphs")

The dataset consists of 232 paragraphs


In [ ]:
for paragraph in dataset[:10]:
  formatted_paragraph=textwrap.fill(paragraph)
  print(f"{formatted_paragraph}")

The Lagos air was thick with humidity, but the energy in the club was
electric. The band launched into a hypnotic Afrobeat groove, the drums
pounding out a complex polyrhythm, the horns blaring a soaring melody,
and the bass laying down a deep, funky foundation. A woman named Imani
moved effortlessly to the music, her body swaying in time with the
rhythm. The music seemed to flow through her, a powerful current of
energy and joy. All around her, people were dancing, singing, and
clapping, caught up in the infectious rhythm. The music was more than
just entertainment; it was a celebration of life, a connection to
their shared heritage, a vibrant expression of the soul of Lagos.
The warm evening air in Accra was filled with the lilting melodies of
Highlife music. At a small bar tucked away on a side street, a band
played, the guitars weaving intricate patterns, the horns adding a
bright, joyful counterpoint. Kwame, a man with a wistful smile, sat at
a table nursing a beer, lost in the mu

We have to split the paragraphs into words and this process is called tokenization.And the simplesst tokenizer is called space tokenizer.This breaks sentences into individual characters that are produced by pressing the space bar

In [ ]:
def space_tokenizer(text:str)->list[str]:
  tokens=text.split(" ")
  return tokens


In [ ]:

space_tokenizer("The Lord is my Light,my Joy and my song")

['The', 'Lord', 'is', 'my', 'Light,my', 'Joy', 'and', 'my', 'song']

In [ ]:
space_tokenizer(dataset[0])

['The',
 'Lagos',
 'air',
 'was',
 'thick',
 'with',
 'humidity,',
 'but',
 'the',
 'energy',
 'in',
 'the',
 'club',
 'was',
 'electric.',
 'The',
 'band',
 'launched',
 'into',
 'a',
 'hypnotic',
 'Afrobeat',
 'groove,',
 'the',
 'drums',
 'pounding',
 'out',
 'a',
 'complex',
 'polyrhythm,',
 'the',
 'horns',
 'blaring',
 'a',
 'soaring',
 'melody,',
 'and',
 'the',
 'bass',
 'laying',
 'down',
 'a',
 'deep,',
 'funky',
 'foundation.',
 'A',
 'woman',
 'named',
 'Imani',
 'moved',
 'effortlessly',
 'to',
 'the',
 'music,',
 'her',
 'body',
 'swaying',
 'in',
 'time',
 'with',
 'the',
 'rhythm.',
 'The',
 'music',
 'seemed',
 'to',
 'flow',
 'through',
 'her,',
 'a',
 'powerful',
 'current',
 'of',
 'energy',
 'and',
 'joy.',
 'All',
 'around',
 'her,',
 'people',
 'were',
 'dancing,',
 'singing,',
 'and',
 'clapping,',
 'caught',
 'up',
 'in',
 'the',
 'infectious',
 'rhythm.',
 'The',
 'music',
 'was',
 'more',
 'than',
 'just',
 'entertainment;',
 'it',
 'was',
 'a',
 'celebration

In [ ]:
all_unigrams=[]
all_bigrams=[]
all_trigrams=[]

def generate_ngrams(text:str,n:int)->list[tuple[str]]:
  tokens=space_tokenizer(text)

  ngrams=[]

  num_of_tokens=len(tokens)

  for i in range(0,num_of_tokens- n + 1):
    ngrams.append(tuple(tokens[i:i+n]))
  return ngrams


In [ ]:
for paragraph in dataset:
  all_unigrams.extend(generate_ngrams(paragraph,n=1))
  all_bigrams.extend(generate_ngrams(paragraph,n=2))
  all_trigrams.extend(generate_ngrams(paragraph,n=3))

print("First 10 Unigrams:",all_unigrams[:10])
print("First 10 Bigrams:",all_bigrams[:10])
print("First 10 Trigrams:",all_trigrams[:10])

First 10 Unigrams: [('The',), ('Lagos',), ('air',), ('was',), ('thick',), ('with',), ('humidity,',), ('but',), ('the',), ('energy',)]
First 10 Bigrams: [('The', 'Lagos'), ('Lagos', 'air'), ('air', 'was'), ('was', 'thick'), ('thick', 'with'), ('with', 'humidity,'), ('humidity,', 'but'), ('but', 'the'), ('the', 'energy'), ('energy', 'in')]
First 10 Trigrams: [('The', 'Lagos', 'air'), ('Lagos', 'air', 'was'), ('air', 'was', 'thick'), ('was', 'thick', 'with'), ('thick', 'with', 'humidity,'), ('with', 'humidity,', 'but'), ('humidity,', 'but', 'the'), ('but', 'the', 'energy'), ('the', 'energy', 'in'), ('energy', 'in', 'the')]


In [ ]:
ngrams.test_generate_ngrams(generate_ngrams,space_tokenizer)

✅ Nice! Your implementation looks correct.


In [ ]:
# Convert any list-like elements to standard tuples to make them hashable
all_bigrams_clean = [tuple(item) if isinstance(item, (list, tuple)) else item for item in all_bigrams]
all_trigrams_clean = [tuple(item) if isinstance(item, (list, tuple)) else item for item in all_trigrams]

bigram_counts = Counter(all_bigrams_clean)
print("Most common bigrams:")
for bigram, count in bigram_counts.most_common(10):
  print(f" ({bigram},{count})")

trigram_counts = Counter(all_trigrams_clean)
print("\nMost common trigrams:")
for trigram, count in trigram_counts.most_common(10):
  print(f" ({trigram},{count})")

Most common bigrams:
 (('is', 'a'),144)
 (('of', 'the'),100)
 (('and', 'the'),69)
 (('in', 'the'),61)
 (('with', 'a'),60)
 (('in', 'a'),55)
 (('and', 'a'),50)
 (('to', 'the'),42)
 (('was', 'a'),39)
 (('It', 'is'),33)

Most common trigrams:
 (('went', 'looking', 'for'),32)
 (('a', 'symbol', 'of'),18)
 (('was', 'hungry', 'so'),18)
 (('The', 'result', 'is'),17)
 (('looking', 'for', 'a'),17)
 (('she', 'went', 'looking'),16)
 (('he', 'went', 'looking'),16)
 (('result', 'is', 'a'),15)
 (('so', 'he', 'went'),14)
 (('so', 'she', 'went'),14)


In [ ]:
def get_ngram_counts(dataset:list[str],n:int)->dict[str,Counter]:
  ngrams_counts=defaultdict(Counter)

  for paragraph in dataset:
    for ngram in generate_ngrams(paragraph,n):
      context=" ".join(ngram[:-1])
      next_token=ngram[-1]
      ngrams_counts[context][next_token]+=1
  return dict(ngrams_counts)

In [ ]:
example_data = [
    "This is an example sentence.",
    "Another example sentence.",
    "Split a sentence."
]

ngram_counts=get_ngram_counts(example_data,2)

print("Bigram counts dictionary:\n")
print("{")
for context,counter in ngram_counts.items():
  print(f"  {context}: {dict(counter)}")
print("}")

Bigram counts dictionary:

{
  This: {'is': 1}
  is: {'an': 1}
  an: {'example': 1}
  example: {'sentence.': 2}
  Another: {'example': 1}
  Split: {'a': 1}
  a: {'sentence.': 1}
}


In [ ]:
ngrams.test_ngram_counts(get_ngram_counts,generate_ngrams)

✅ Nice! Your implementation looks correct.


In [ ]:
bigram_counts = get_ngram_counts(dataset, n=2)

bigram_counts_matrix = {
    context: dict(counts) for context, counts in bigram_counts.items()
}
bigram_data_frame = pd.DataFrame.from_dict(
    bigram_counts_matrix, orient="index").fillna(0)

display(bigram_data_frame)

zero_count = (bigram_data_frame == 0).sum().sum()
print(
    f"Number of bigrams with a count of 0: {zero_count:,}"
    f" ({zero_count/bigram_data_frame.size * 100:.2f}%)"
)

,Lagos,band,music,warm,Highlife,bustling,Dakar,Mbalax,Kinshasa,Soukous,...,"kudu,","mph),",Ostriches,Antarctic,plumage,surface.,(Spheniscus,demersus).,breed,Bay
The,1.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
of,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
the,0.0,1.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
a,0.0,1.0,0.0,6.0,0.0,7.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
with,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
water's,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Penguin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
(Spheniscus,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
penguins,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Number of bigrams with a count of 0: 26,606,550 (99.95%)


In [ ]:
trigram_counts=get_ngram_counts(dataset,n=3)
trigram_counts_matrix={
    context:dict(counts) for context,counts in trigram_counts.items()
}
trigram_data_frame=pd.DataFrame.from_dict(
    trigram_counts_matrix,orient="index").fillna(0)
display(trigram_data_frame)

zero_count=(trigram_data_frame==0).sum().sum()
print(
    f"Number of trigrams with a count of 0: {zero_count:,}"
    f" ({zero_count/trigram_data_frame.size * 100:.2f}%)"
)

,air,was,thick,thin,always,"quiet,",filled,alive,with,"humidity,",...,plumage,water's,surface.,penguin,(Spheniscus,demersus).,penguins,breed,Algoa,Bay
The Lagos,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
in the,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
and the,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
warm evening,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
vegetables. The,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Penguin (Spheniscus,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
demersus). These,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
These penguins,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
down to,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Number of trigrams with a count of 0: 68,942,324 (99.98%)


In [ ]:

trigram_counts=get_ngram_counts(dataset,n=3)
bigram_counts=get_ngram_counts(dataset,n=2)

In [ ]:
context="a staple"
trigram_counts[context]

Counter({'food': 1,
         'in': 6,
         'dish': 2,
         'throughout': 1,
         'of': 1,
         'at': 1,
         'beverage': 1})

In [ ]:
context="a staple"
bigram_count_a_staple=sum(trigram_counts[context].values())

print(
    "Bigram count of 'a staple' computed indirectly from trigram counts: ",
    bigram_count_a_staple,
)
print("Bigram count of 'a staple' computed directly from bigram counts: ",
bigram_counts["a"]["staple"])

Bigram count of 'a staple' computed indirectly from trigram counts:  13
Bigram count of 'a staple' computed directly from bigram counts:  13


In [ ]:
def build_ngram_model(dataset:list[str],n:int)-> dict[str,dict[str,float]]:
  ngram_counts=get_ngram_counts(dataset,n)

  ngram_model={}
  ngram_counts=get_ngram_counts(dataset,n)

  for context,next_tokens in ngram_counts.items():
    context_total_count=sum(next_tokens.values())
    ngram_model[context]={}
    for token,count in next_tokens.items():
      ngram_model[context][token]=count/context_total_count
  return ngram_model

In [ ]:
ngrams.test_build_ngram_model(build_ngram_model,get_ngram_counts)

✅ Nice! Your implementation looks correct.


In [ ]:
trigram_model=build_ngram_model(dataset,n=3)


In [ ]:
print(f"P(B|\"as it\")={trigram_model['as it']}")
print(f"P(B|\"as they\")={trigram_model['as they']}")

P(B|"as it")={'is': 0.6666666666666666, 'receives': 0.3333333333333333}
P(B|"as they")={'were': 1.0}


In [ ]:
context="The name"
trigram_model[context]

{'means': 0.6666666666666666, "'Etosha'": 0.3333333333333333}

In [ ]:
example_candidate_tokens=['apple','banana','cherry']
probabilites=[0.2,0.5,0.3]

chosen_fruit=random.choices(
    example_candidate_tokens,
    weights=probabilites,
    k=1
)[0]
print(f"Chosen fruit: {chosen_fruit}")

Chosen fruit: banana


In [ ]:
context="looking for"
candidate_tokens=[]
candidate_tokens_probabilities=[]



for token,prob in trigram_model[context].items():
  candidate_tokens.append(token)
  candidate_tokens_probabilities.append(prob)

print(f"Candidate tokens:{candidate_tokens}")
print(f"Probabilities:{candidate_tokens_probabilities}")

next_token=random.choices(
    candidate_tokens,
    weights=candidate_tokens_probabilities)[0]
print("\n\nSampled next token:")
print(context,next_token)

Candidate tokens:['the', 'a', 'Banku', 'Tella,', 'Maafe,', 'Umqombothi,', 'sugarcane', 'crispy', 'warm', 'Doro', 'sambusa,', 'dodo,', 'Fura']
Probabilities:[0.125, 0.53125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125]


Sampled next token:
looking for a


In [ ]:
ngrams.test_candidate_tokens(
    trigram_model,candidate_tokens,candidate_tokens_probabilities
)

✅ Nice! Your answer looks correct.


In [ ]:
def generate_next_n_tokens(
    n: int,
    ngram_model: dict[str, dict[str, float]],
    prompt: str,
    num_tokens_to_generate: int,
) -> str:

    # Split prompt into individual tokens.
    generated_words = space_tokenizer(prompt)

    for _ in range(num_tokens_to_generate):
        # Get last (n-1) tokens as context.
        context = generated_words[-(n - 1):]
        context = " ".join(context)
        if context in ngram_model:
            # Sample next word based on probabilities.
            next_word = random.choices(
                list(ngram_model[context].keys()),
                weights=ngram_model[context].values()
            )[0]

            generated_words.append(next_word)
        else:
            print(
                "⚠️ No valid continuation found. Change the prompt or"
                " try sampling another continuation.\n"
            )
            break

    return " ".join(generated_words)

In [ ]:
prompt="Jide was hungry so she went looking for"
bigram_model=build_ngram_model(dataset,n=2)

n=2
num_tokens_to_generate=10
generate_next_n_tokens(
    n=n,
    ngram_model=bigram_model,
    prompt=prompt,
    num_tokens_to_generate=num_tokens_to_generate
)

'Jide was hungry so she went looking for a basalt cliff into a plate of North Africa. It'

In [ ]:
prompt="Jide was hungry so she went looking for"
n=3
num_tokens_to_generate=10
generate_next_n_tokens(
    n=n,
    ngram_model=trigram_model,
    prompt=prompt,
    num_tokens_to_generate=num_tokens_to_generate,
)

'Jide was hungry so she went looking for the kings of the sea provided a burst of flavor'